# Matrix Factorisation on Movie Lens 1M dataset
Dataset from: [Movie Lens 1M Dataset](https://grouplens.org/datasets/movielens/1m/)

In [ ]:
# Standard
import pandas as pd
import re

# Third-party
import numpy as np
import plotly.express as px
from sklearn.metrics import explained_variance_score, mean_squared_error, mean_absolute_error, r2_score

# Local
from matrix_factorisation import MatrixFactorisation

### Load dataset

In [ ]:
def load_movielens_movies(path: str = "Dataset/ml-1m/movies.dat") -> pd.DataFrame:
    return pd.read_csv(
        path,
        sep="::",
        engine="python",
        names=["movie_id", "title", "genres"],
        encoding="latin-1",
    )

def load_movielens_ratings(path: str = "Dataset/ml-1m/ratings.dat") -> pd.DataFrame:
    return pd.read_csv(
        path,
        sep="::",
        engine="python",
        names=["user_id", "movie_id", "rating", "timestamp"],
        encoding="latin-1",
    )

def preprocess_data(split_titles=False):
    """
    Merges movies and ratings on movieId.
    """
    movies = load_movielens_movies()
    # Split title and year out of title
    if split_titles:
        movies["year"] = movies["title"].apply(lambda movie_name: re.search('\\((\\d*)\\)', movie_name).groups(1)[0])
        movies["title"] = movies["title"].apply(lambda movie_name: movie_name.split(" (")[0])
        # TODO - fix what happens for duplicated movie titles
    # Get ratings
    ratings = load_movielens_ratings()
    merged = pd.merge(ratings, movies, on="movie_id")
    # Convert timestamps from unix
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], unit='s')
    return merged

In [ ]:
df = preprocess_data()

In [ ]:
df.head()

### EDA

Ref:
- [MovieLens-1M Deep Dive – Part I](https://towardsdatascience.com/movielens-1m-deep-dive-part-i-8acfeda1ad4/)

In [ ]:
df.rating.value_counts().sort_index().plot(kind="bar", xlabel="Rating", ylabel="Count")

In [ ]:
df.timestamp.groupby(df.timestamp.dt.year).count().plot(kind="bar", xlabel="Rating Timestamp", ylabel="Count")

In [ ]:
movies_df = df.copy(deep=True)

In [ ]:
movies_df['genres'] = movies_df['genres'].apply(lambda x: x.split('|'))
movies_df_exploded = movies_df.explode("genres")
px.histogram(movies_df_exploded, x="genres",
             height=400, width=800, title="Movie count by genre").update_xaxes(categoryorder="total descending")

In [ ]:
rating_by_genre_df = movies_df_exploded.groupby('genres').agg({'rating': ['mean', 'count']}).sort_values(('rating', 'mean')).reset_index()  # noqa: E501
rating_by_genre_df.columns = ['_'.join(col).strip() for col in rating_by_genre_df.columns.values]
px.bar(rating_by_genre_df, x='genres_', y='rating_mean', height=300, width=800)

In [ ]:
movies_df["year"] = movies_df["title"].apply(lambda movie_name: re.search('\\((\\d*)\\)', movie_name).groups(1)[0])
movie_count_by_year = px.histogram(movies_df, x='year',
                                   height=400, width=800,
                                   title='Movie count by year').update_xaxes(categoryorder="category ascending")
movie_count_by_year

### Matrix Factorisation
Matrix factorisation algorithm applied to Top-N and Similarity (by movie) slates.

i.e. answers the questions: "what are the top N movies for a specific user" and "because someone watched a movie, they should watch"


Using:
- [SKLearn NMF (Non-Negative Matrix Factorisation)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)]
- Similarity code from [here](https://github.com/dinesh-git17/movie_recommendation/tree/main)
- Top-N code from [here](https://medium.com/@quindaly/step-by-step-nmf-example-in-python-9974e38dc9f9)

In [ ]:
NMF_model = MatrixFactorisation(df, n_components=50)

In [ ]:
recs = NMF_model.movie_similarity("101 Dalmatians (1961)")
recs

In [ ]:
recs = NMF_model.movie_similarity("101 Dalmatians (1996)")
recs

In [ ]:
recs = NMF_model.movie_similarity("10 Things I Hate About You (1999)")
recs

In [ ]:
recs = NMF_model.movie_similarity("Young Guns (1988)")
recs

Thoughts:
* Not recommending sequels
* Not using a test-train split -> this algorithm won't work if the requested movie doesn't exist in the pivot table
* Therefore, can't handle new movies or users

## Top-N movies for user

In [ ]:
user_id  = 44
NMF_model.understand_user_profile(user_id)
rec = NMF_model.user_top_N(user_id)
print(f"Recommendations for user 44:")
display(NMF_model.get_recommend_dataframe(rec))

### Evaluation metrics
Offline metrics - adapted from [here](https://github.com/aryan-jadon/Evaluation-Metrics-for-Recommendation-Systems/blob/main/recommenders/evaluation/python_evaluation.py)

Table from [here](https://github.com/recommenders-team/recommenders/blob/main/examples/03_evaluate/evaluation.ipynb)
|Metric|Range|Selection criteria|Limitation|Reference|
|------|-------------------------------|---------|----------|---------|
|RMSE|$> 0$|The smaller the better.|May be biased, and less explainable than MAE|[link](https://en.wikipedia.org/wiki/Root-mean-square_deviation)|
|MAE|$\geq 0$|The smaller the better.|Dependent on variable scale.|[link](https://en.wikipedia.org/wiki/Mean_absolute_error)|
|R2|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Coefficient_of_determination)|
|Explained variance|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Explained_variation)|

In [ ]:
def evaluation_metrics(y_true, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R Squared": r2_score(y_true, y_pred),
        "Explained variance": explained_variance_score(y_true, y_pred)
    }

In [ ]:
[print(f"{k}: {v:.2f}") for k, v in evaluation_metrics(NMF_model.pivot, NMF_model.V).items()]

## User profile evaluation

In [ ]:
user_ids = [6013, 2195, 1198, 3662, 4713]

In [ ]:
for user_id in user_ids:
    NMF_model.understand_user_profile(user_id, rating_dist=False, wc=False)
    rec = NMF_model.user_top_N(user_id)
    print(f"Recommendations for user {user_id}")
    display(NMF_model.get_recommend_dataframe(rec))